# Suite REF — References: one format for objects, relation entries and frames

A reference designates what a request or a response points at ([CAP-130: references to objects, entries and frames](../../../mbse-basis/docs/capabilities/CAP-130.md);
FRAMEWORK.md, Resolved, References): `References.OfObject`, an object, by its path (a singleton, an owned object or a
schema object), never a counted one, which nothing names (object identity 40 and 55 in [CAP-118: singletons, owned and counted objects](../../../mbse-basis/docs/capabilities/CAP-118.md#decided));
`References.OfEntry`, a relation entry, by its relation's path and all its links and properties; `References.OfFrame`,
a frame, each symbol's object and each variable's value, and what it witnesses, objects' property values
(`References.OfProperties`) and relation entries, each by its objects' references. `References.of(store, x)`
gives one, a value of the store's, written and read as any value is; `References.resolve(store, reference)` gives what
it designates, or None where the store holds none.

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
import json

from mbse.Schemas.Framework import JSON, Builders as B, Paths, Plain, References
from support import raises


def crm(txn):
    """crm.Person, who knows persons since a year; crm.Book, a singleton owning its contacts by key; crm.Note, counted,
    about a person."""
    S = txn.Schemas
    return txn.uid("crm/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Knows": S.OfRelation, "Person": S.OfObject, "Holds": S.OfRelation, "Book": S.OfObject,
                  "Note": S.OfObject, "About": S.OfRelation})
        .spec([
            lambda _: _.Knows.name("crm.Knows").links([lambda t: t.name("knower"), lambda t: t.name("known")]).properties({
                "since": S.OfInt}),
            lambda _: _.Person.name("crm.Person").ref(True).properties({"name": S.OfStr}).adjacencies([
                lambda t: t.name("knows").relation([_.Knows]).me("knower"),
                lambda t: t.relation([_.Knows]).me("known"),
                lambda t: t.name("book").relation([_.Holds]).me("contact").owner(True),
            ]).adjacencies({"trio": lambda t: t.relation([S.OfRelation(lambda r: r.links([  # no name of its own
                lambda t: t.name("a"), lambda t: t.name("b"), lambda t: t.name("c")]))]).me("a")}),
            lambda _: _.Holds.name("crm.Holds").links([lambda t: t.name("book"), lambda t: t.name("contact")]).properties({
                "key": S.OfStr}).uniques([["book", "key"], ["contact"]]),
            lambda _: _.Book.name("crm.Book").ref(True).singleton("crm.TheBook").adjacencies({
                "contacts": lambda t: t.relation([_.Holds]).me("book")}),
            lambda _: _.Note.name("crm.Note").ref(True).properties({"text": S.OfStr}).adjacencies({
                "about": lambda t: t.relation([_.About]).me("note")}),
            lambda _: _.About.name("crm.About").links([lambda t: t.name("note"), lambda t: t.name("person")]),
        ])]))])


store = B.Store()
store.transaction(crm)
Person, Book, Note = (store.registered(name) for name in ("crm.Person", "crm.Book", "crm.Note"))
book = store.singleton("crm.TheBook")
frames = store.transaction(lambda t: t.uid("people").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
    .symbols({"book": Book, "ada": Person, "bob": Person, "cy": Person}).spec([
        lambda _: _.book.contacts({"ada": lambda e: e.contact(_.ada), "bob": lambda e: e.contact(_.bob),
                                   "cy": lambda e: e.contact(_.cy)}),
        lambda _: _.ada.name("Ada").knows([lambda e: e.known(_.bob).since(2020), lambda e: e.known(_.cy).since(2021)]),
        lambda _: _.bob.name("Bob"),
        lambda _: _.cy.name("Cy"),
    ])]))]))
ada, bob, cy = (frames[0][name] for name in ("ada", "bob", "cy"))


def notes(t):
    """A note about each person, upserted with the key `notes/1` for the frame that reaches it."""
    return t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols({"contact": Person})])
        .then([lambda n: n.upsert(lambda u: u
        .key("notes/1").exists([lambda p: p.symbols({"contact": Person, "note": Note}).spec([
            lambda _: _.note.text("hi").about([_.contact])])]))]))])


def plain(reference):
    return Plain.ToPlain(store)(reference.as_schema(), reference)


def path(singleton, *steps):
    """A path's plain form: from a singleton through entries, each (i, o, keys)."""
    nested = {"target": {}}
    for i, o, keys in reversed(steps):
        nested = {"entry": {"i": i, "o": o, "properties": keys, "next": nested}}
    return {"singleton": singleton, "next": nested}

## REF-01 · Every store holds the reference schemas, its own; References.install, run again, writes nothing

In [ ]:
OfObject = store.registered("References.OfObject")
OfFrame, OfEntry = store.registered("References.OfFrame"), store.registered("References.OfEntry")
assert list(OfObject.properties) == ["path"] and OfObject.properties["path"].schema is store.registered("Paths.OfRoot")
OfProperties = store.registered("References.OfProperties")
assert list(OfFrame.properties) == ["binds", "variables", "assigns", "entries"]
assert OfFrame.properties["binds"].schema.element is OfObject and OfFrame.properties["assigns"].schema.element is OfProperties
assert OfFrame.properties["entries"].schema.element is OfEntry and list(OfProperties.properties) == ["object", "properties"]
assert list(OfEntry.properties) == ["relation", "links", "properties"]
assert OfEntry.properties["relation"].schema is store.registered("Paths.OfRoot")
assert OfEntry.properties["links"].schema.element is OfObject
assert not any(schema.ref for schema in (OfObject, OfFrame, OfProperties, OfEntry))  # value objects, as a path is
other = B.Store()
assert other.registered("References.OfObject") is not OfObject  # each store's its own
names = {name: store.registered(name) for name in store.names()}
store.transaction(References.install)  # each found by its name
assert {name: store.registered(name) for name in store.names()} == names

## REF-02 · An object's reference by its path: a singleton, an owned object, a schema object; each resolved

In [ ]:
assert plain(References.of(store, book)) == {"path": path("crm.TheBook")}
assert plain(References.of(store, ada)) == {"path": path("crm.TheBook", ("contacts", "book", {"key": "ada"}))}
assert plain(References.of(store, Person)) == {"path": path(".", ("schemas", "", {"name": "crm"}),
                                                              ("schemas", "", {"name": "Person"}))}
for target in (book, ada, Person):
    assert References.resolve(store, References.of(store, target)) is target
built = store.References.OfObject(lambda o: o.path(lambda p: p.singleton("crm.TheBook").next(lambda n: n.entry(
    lambda e: e.i("contacts").o("book").properties({"key": "bob"}).next(lambda n: n.target(lambda t: t))))))
assert References.resolve(store, built) is bob  # written through the builders' fluent calls

## REF-03 · A counted object has no reference, nor a frame that binds one: an upsert's, which the executor alone finds again

In [ ]:
made = store.transaction(notes)
assert [frame["contact"] for frame in made] == [ada, bob, cy]
note = made[0]["note"]
with raises(LookupError, match="a counted 'crm.Note' has no reference: nothing names it"):
    References.of(store, note)
with raises(LookupError, match="symbol 'note' of the frame: a counted 'crm.Note' has no reference"):
    References.of(store, made[0])
again = store.transaction(notes)  # run again: the executor finds what it made, which nothing outside it names
assert again[0]["note"] is note

## REF-04 · A relation entry by its relation's path and all its links and properties: a keyed relation's, one without properties; refused where it links a counted object

In [ ]:
held = next(iter(book.contacts))  # crm.Holds, keyed by `key`
assert plain(References.of(store, held)) == {
    "relation": path(".", ("schemas", "", {"name": "crm"}), ("schemas", "", {"name": "Holds"})),
    "links": {"book": {"path": path("crm.TheBook")},
              "contact": {"path": path("crm.TheBook", ("contacts", "book", {"key": "ada"}))}},
    "properties": {"key": {"str": "ada"}}}
knows = list(ada.knows)
assert plain(References.of(store, knows[1]))["properties"] == {"since": {"int": 2021}}
with store.transaction():
    store.crm.Person(bob, lambda b: b.knows(lambda a: a.insert(lambda e: e.known(cy))))  # since left unset
bare = next(iter(bob.knows))
assert plain(References.of(store, bare))["properties"] == {}
with raises(LookupError, match="a counted 'crm.Note' has no reference"):  # crm.About links a note
    References.of(store, next(iter(note.about)))
for entry in (held, knows[0], knows[1], bare):
    assert References.resolve(store, References.of(store, entry)) is entry
changed = store.References.OfEntry(lambda e: e.relation(Paths.of(store.registered("crm.Knows"))).links({
    "knower": lambda o: o.path(Paths.of(ada)), "known": lambda o: o.path(Paths.of(cy))}).properties({"since": 1999}))
assert References.resolve(store, changed) is None  # a property no entry has
fewer = store.References.OfEntry(lambda e: e.relation(Paths.of(store.registered("crm.Knows"))).links({
    "knower": lambda o: o.path(Paths.of(ada))}))
assert References.resolve(store, fewer) is None  # every link, or none
assert References.resolve(store, store.References.OfEntry(lambda e: e.relation(Paths.of(store.registered("crm.Knows"))))) \
    is None  # no links

## REF-05 · A frame a transaction gave back: each symbol's object, resolved to the frame

In [ ]:
frame = frames[0]
reference = References.of(store, frame)
assert sorted(plain(reference)["binds"]) == sorted(frame)
assert plain(reference)["binds"]["ada"] == plain(References.of(store, ada))
assert References.resolve(store, reference) == frame
assert plain(References.of(store, {})) == {"binds": {}} and References.resolve(store, References.of(store, {})) == {}
assert References.resolve(store, store.References.OfFrame(lambda f: f)) == {}  # no binds: the empty frame

## REF-06 · What is refused: a counted object nothing names, another store's object, what is not an object, an entry or a frame

In [ ]:
with store.transaction():
    loose = store.crm.Note(lambda n: n.text("by hand"))  # counted, written outright: nothing names it
with raises(LookupError, match="a counted 'crm.Note' has no reference: nothing names it"):
    References.of(store, loose)
with raises(LookupError, match="symbol 'loose' of the frame: a counted 'crm.Note' has no reference"):
    References.of(store, {"loose": loose})
with raises(TypeError, match="the object belongs to another store"):
    References.of(other, ada)
with raises(TypeError, match="References.of takes an object, a relation entry or a frame, not 'x'"):
    References.of(store, "x")
with raises(TypeError, match="References.of takes an object, a relation entry or a frame, not 'x'"):
    References.of(store, {"x": "x"})  # a frame binding what is not an object
with store.transaction():
    store.crm.Person(ada, lambda a: a.trio(lambda t: t.insert(lambda e: e.b(bob).c(cy))))
with raises(LookupError, match="has no path"):  # an entry of a relation with no name of its own
    References.of(store, next(iter(ada.trio)))
for wrong in ("x", Paths.of(ada)):
    with raises(TypeError, match="References.resolve takes a References.OfObject, OfEntry or OfFrame value of the store's"):
        References.resolve(store, wrong)

## REF-07 · A reference written and read back as a value; None where the store holds nothing it designates

In [ ]:
held = next(iter(book.contacts))  # as the store holds it now: an update of ada's rewrote her entries
for target in (ada, held, frame):
    reference = References.of(store, target)
    text = JSON.ToJSON(store)(reference.as_schema(), reference)
    back = JSON.FromJSON(store)(reference.as_schema(), text)
    found = References.resolve(store, back)
    assert found == target if isinstance(target, dict) else found is target
empty = B.Store()  # its schemas and nothing else
empty.transaction(crm)
for target in (ada, held, frame, Person):
    reference = References.of(store, target)
    back = JSON.FromJSON(empty)(empty.registered(store.name_of(reference.as_schema())),
                                JSON.ToJSON(store)(reference.as_schema(), reference))
    assert References.resolve(empty, back) is (empty.registered("crm.Person") if target is Person else None)
bare = B.Store()  # not even the schemas
reference = References.of(store, held)
assert References.resolve(bare, JSON.FromJSON(bare)(bare.References.OfEntry.as_schema(),
                                                    JSON.ToJSON(store)(reference.as_schema(), reference))) is None
nowhere = store.References.OfObject(lambda o: o.path(lambda p: p.singleton(".").next(lambda n: n.entry(
    lambda e: e.i("schemas").o("x").properties({"name": "crm"}).next(lambda n: n.target(lambda t: t))))))
gone = store.References.OfEntry(lambda e: e.relation(Paths.of(Person)).links({"knower": lambda o: o.path(Paths.of(ada))}))
assert References.resolve(store, nowhere) is References.resolve(store, gone) is None

## REF-08 · A frame as a witness: property values an object holds, and relation entries, each by its objects' references

In [ ]:
witness = store.References.OfFrame(lambda f: f
    .binds({"ada": lambda o: o.path(Paths.of(ada)), "bob": lambda o: o.path(Paths.of(bob))})
    .assigns([lambda x: x.as_object(lambda p: p.object(lambda o: o.path(Paths.of(ada))).properties({"name": "Ada"}))])
    .entries([lambda x: x.as_object(lambda e: e.relation(Paths.of(store.registered("crm.Knows"))).links({
        "knower": lambda o: o.path(Paths.of(ada)), "known": lambda o: o.path(Paths.of(bob))}).properties({"since": 2020}))]))
assert plain(witness)["assigns"] == [{"object": plain(References.of(store, ada)), "properties": {"name": {"str": "Ada"}}}]
assert plain(witness)["entries"] == [plain(References.of(store, next(iter(ada.knows))))]  # an OfEntry, as of() gives one
text = JSON.ToJSON(store)(witness.as_schema(), witness)
assert plain(JSON.FromJSON(store)(witness.as_schema(), text)) == plain(witness)  # carried, and read back the same
assert References.resolve(store, witness) == {"ada": ada, "bob": bob}  # resolved: its bindings

## REF-09 · A pattern's symbol given a reference in place of a schema: it binds what the reference designates, one frame, or none; its spec filters or is written as any symbol's, by the schema its path implies where the store holds nothing it designates; a symbol has exactly one of a schema and a reference

In [ ]:
reference = References.of(store, ada)  # a symbol given a reference binds what it designates, one frame each
known = store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"x": reference, "y": Person}).spec([lambda _: _.x.knows([_.y])])]))]))
assert [(f["x"], str(f["y"].name)) for f in known] == [(ada, "Bob"), (ada, "Cy")]  # from one object, along an adjacency
assert [f["y"] for f in store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"x": References.of(store, bob), "y": Person}).spec([lambda _: _.x.knows([_.y])])]))]))] == [cy]
assert store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"x": reference}).spec([lambda _: _.x.name("Bob")])]))])) == []  # its spec filters, as any symbol's
upserted = store.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
    .symbols({"x": References.of(store, cy)}).spec([lambda _: _.x.name("Cy C.")])]))]))
assert upserted == [{"x": cy}] and str(cy.name) == "Cy C."  # an upsert writes its spec over what it binds
assert store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"s": References.of(store, Person)}).spec([lambda _: _.s.name("crm.Person")])]))])) == [{"s": Person}]
nowhere = store.References.OfObject(lambda o: o.path(lambda p: p.singleton("crm.TheBook").next(lambda n: n.entry(
    lambda e: e.i("contacts").o("book").properties({"key": "zed"}).next(lambda n: n.target(lambda t: t))))))
for term in (lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols({"x": nowhere})])),
             lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols({"x": nowhere})]))):
    assert store.transaction(lambda t: t.mutations([term])) == []  # it designates nothing: no frame, nothing made
assert store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"x": nowhere}).spec([lambda _: _.x.name("Zed")])]))])) == []  # its spec, by the schema its path implies


def at(held, singleton, *steps):
    """A `References.OfObject` of `held`'s, from a singleton through entries, each (i, o, keys)."""
    def nested(n, rest):
        if not rest:
            return n.target(lambda t: t)
        (i, o, keys), *more = rest
        return n.entry(lambda e: e.i(i).o(o).properties(keys).next(lambda n: nested(n, more)))
    return held.References.OfObject(lambda o: o.path(lambda p: p.singleton(singleton).next(lambda n: nested(n, steps))))


client = B.Store()  # the schemas, not the data: its spec is written by the schemas its reference's path goes through
client.transaction(crm)
sent = client.Mutations.OfTransaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"x": at(client, "crm.TheBook", ("contacts", "book", {"key": "ada"})), "y": client.registered("crm.Person")})
    .spec([lambda _: _.x.knows([_.y])])]))]))
text = json.dumps(Plain.ToPlain(client).Transaction(sent))  # written as JSON, read where the data is, and run
assert store.transaction(Plain.FromPlain(store).Transaction(json.loads(text))) == known  # the frames, as given above


def firm(txn):
    """crm.Firm, owned through crm.Holds as a person is, by an adjacency also named `book`."""
    S = txn.Schemas
    return txn.uid("firm/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Holds": S.OfRelation, "Firm": S.OfObject}).spec([
            lambda _: _.Holds.name("crm.Holds"),
            lambda _: _.Firm.name("crm.Firm").ref(True).adjacencies([
                lambda t: t.name("book").relation([_.Holds]).me("contact").owner(True)]),
        ])]))])


firms = B.Store()
firms.transaction(crm)
firms.transaction(firm)
for held, given, why in (
        (client, at(client, "crm.Nope"), "starts from 'crm.Nope', which no schema declares a singleton"),
        (client, at(client, "crm.TheBook", ("nope", "book", {})),
         "steps through 'nope', an adjacency 'crm.Book' does not declare"),
        (client, at(client, "crm.TheBook", ("contacts", "nope", {})), "steps through 'contacts' of 'crm.Book', where not "
         "exactly one schema declares an adjacency 'nope' marked owner() on another link of its relation: found []"),
        (firms, at(firms, "crm.TheBook", ("contacts", "book", {"key": "ada"})), "steps through 'contacts' of 'crm.Book', "
         "where not exactly one schema declares an adjacency 'book' marked owner() on another link of its relation: "
         "found ['crm.Firm', 'crm.Person']"),
        (client, at(client, ".", ("schemas", "", {"name": "Nope"})), "names a schema object the store does not hold")):
    message = f"symbol 'x' is given a reference whose path {why}: a spec writes what its object's schema declares"
    with raises(LookupError, match=message), held.transaction():
        held.Schemas.OfPattern(lambda p: p.symbols({"x": given}).spec([lambda _: _.x.name("Zed")]))
with store.transaction():
    both = store.Schemas.OfPattern(lambda p: p.symbols([lambda s: s.name("x").schema([Person]).reference(reference)]))
    neither = store.Schemas.OfPattern(lambda p: p.symbols([lambda s: s.name("x")]))
    given = store.Schemas.OfPattern(lambda p: p.symbols({"x": reference}))
assert References.validate(both) == ["symbol 'x' has both a schema and a reference: a symbol has exactly one"]
assert References.validate(neither) == ["symbol 'x' has neither a schema nor a reference: a symbol has exactly one"]
assert References.validate(given) == [] and given.symbols["x"].schema is None
with raises(TypeError, match="References.validate takes a pattern, not 'x'"):
    References.validate("x")
with raises(TypeError, match="symbol 'x' has both a schema and a reference: a symbol has exactly one"), \
        store.transaction():
    store.Schemas.OfPattern(lambda p: p.symbols([lambda s: s.name("x").schema([Person]).reference(reference)]).spec([
        lambda _: _.x.name("Ada")]))